# Five Bad Rows Ruin Least Squares

**Topic:** Robust regression &nbsp;·&nbsp; **Difficulty:** Hard

A pressure sensor glitched five times during the run and reported values a
thousand times too large. Squared error takes those five rows and squares their
influence, so ordinary least squares bends the whole fit toward them.

Huber loss is quadratic near zero and linear in the tails, which caps how much
any single row can pull. It has no closed form, but iteratively reweighted least
squares gets there in a handful of steps.

## The data (created in the next cell)

- `X` `(400, 6)`, `y` `(400,)`, generated from known coefficients `w_true`.
- Exactly 5 rows of `y` are corrupted by a factor of about 1000.
- `w_true` is defined in the data cell above. Do not use it to fit, only to score.

## Your task

With **NumPy only**:

1. Fit OLS in closed form and call the coefficients `w_ols`.
2. Fit Huber regression by IRLS with `delta = 1.35`:
   start from the OLS fit, then repeat 60 times: compute residuals `r`, set
   weights `u = where(|r| <= delta, 1, delta/|r|)`, and solve the weighted normal
   equations `(X^T diag(u) X) w = X^T diag(u) y`.
3. Set `answers` to a dict with `"w_ols"` `(6,)`, `"w_huber"` `(6,)`,
   `"err_ols"` and `"err_huber"`, the L2 distances from each to the true
   coefficients `w_true` from the data cell above.

## Evaluation

Coefficients are compared to the reference to a relative tolerance of 1e-6, and
the evaluation cell checks Huber's error is more than 10x smaller than OLS's.

In [ ]:
import numpy as np

rng = np.random.default_rng(404)
n, d = 400, 6
X = rng.normal(0, 1, (n, d))
w_true = np.array([3.0, -1.5, 0.8, 2.2, -0.4, 1.1])
y = X @ w_true + rng.normal(0, 0.5, n)

bad = rng.choice(n, 5, replace=False)
y[bad] *= 1000.0
print(f"X {X.shape}, corrupted rows: {sorted(bad.tolist())}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("w_ols", [6], 147.5160319822523, 195.86596571617002, 1e-06),
    ("w_huber", [6], 5.2847873568450305, 1.8533426779977078, 1e-06),
    ("err_ols", [], 97.49090034881715, -16.32368281688411, 1e-06),
    ("err_huber", [], 0.060036963775723234, -0.010052470030097096, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "err_ols" in answers and "err_huber" in answers:
    _better = answers["err_huber"] * 10 < answers["err_ols"]
    print(f"  {'huber beats ols 10x':<24} {'ok' if _better else 'wrong'}  "
          f"({answers['err_huber']:.4f} vs {answers['err_ols']:.4f})")
    _ok = _ok and _better

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Each IRLS step is an ordinary weighted least-squares solve, and the weights
`delta/|r|` shrink a corrupted row's influence in proportion to how far out it
is: the five corrupted rows here have |r| between about 220 and 8900, so they get
weights between about 0.006 and 0.00015 instead of the weight 1 that squared
error hands each of them. Starting from OLS is fine here because the algorithm downweights
the outliers within a few iterations.

```python
import numpy as np

w_ols = np.linalg.solve(X.T @ X, X.T @ y)

delta = 1.35
w = w_ols.copy()
for _ in range(60):
    r = y - X @ w
    a = np.abs(r)
    u = np.where(a <= delta, 1.0, delta / np.maximum(a, 1e-12))
    Xu = X * u[:, None]
    w = np.linalg.solve(X.T @ Xu, Xu.T @ y)

w_true_local = np.array([3.0, -1.5, 0.8, 2.2, -0.4, 1.1])
answers = {
    "w_ols": w_ols,
    "w_huber": w,
    "err_ols": float(np.linalg.norm(w_ols - w_true_local)),
    "err_huber": float(np.linalg.norm(w - w_true_local)),
}
```